In [1]:
import numpy as np

# Given logits and ground-truth values
logits = np.array([-2.0, -0.9, -0.5, 0.0, 0.8, 2.2])
truth = np.array([0, 0, 1, 1, 1, 0])

# Convert logits to probabilities using sigmoid
probabilities = 1 / (1 + np.exp(-logits))

# 1. Find the logit corresponding to probability threshold T = 0.3
T = 0.3
logit_T03 = np.log(T / (1 - T))

# 2. Count positive predictions at T = 0.5
n_strict = np.sum(probabilities > 0.5)
n_ge = np.sum(probabilities >= 0.5)

# 3. Calculate Dice score for a given threshold
def calculate_dice(probabilities, truth, threshold):
    predictions = (probabilities > threshold).astype(int)

    tp = np.sum((predictions == 1) & (truth == 1))
    fp = np.sum((predictions == 1) & (truth == 0))
    fn = np.sum((predictions == 0) & (truth == 1))

    denominator = 2 * tp + fp + fn

    if denominator == 0:
        return 1.0

    return (2 * tp) / denominator

dice_03 = calculate_dice(probabilities, truth, 0.3)
dice_05 = calculate_dice(probabilities, truth, 0.5)

# Display results
print("Logits:       ", logits)
print("Probabilities:", np.round(probabilities, 4))
print()
print(f"Logit at T=0.3:       {logit_T03:.4f}")
print(f"Positive pixels (>0.5):  {n_strict}")
print(f"Positive pixels (>=0.5): {n_ge}")
print(f"Dice at T=0.3 (strict):  {dice_03:.4f}")
print(f"Dice at T=0.5 (strict):  {dice_05:.4f}")

# Final answer in the assignment's required format
print("\nFinal Answer:")
print(f"{logit_T03:.4f}, {n_strict}, {n_ge}, {dice_03:.4f}, {dice_05:.4f}")

Logits:        [-2.  -0.9 -0.5  0.   0.8  2.2]
Probabilities: [0.1192 0.2891 0.3775 0.5    0.69   0.9002]

Logit at T=0.3:       -0.8473
Positive pixels (>0.5):  2
Positive pixels (>=0.5): 3
Dice at T=0.3 (strict):  0.8571
Dice at T=0.5 (strict):  0.4000

Final Answer:
-0.8473, 2, 3, 0.8571, 0.4000


In [2]:
# Q2: Per-Class Threshold / Min-Area Selection

configs = {
    "X": {
        "threshold": 0.5,
        "min_area": 0,
        "fg_dice": 0.62,
        "fp_score": 0.9950,
        "empty_accuracy": 0.40
    },
    "Y": {
        "threshold": 0.7,
        "min_area": 100,
        "fg_dice": 0.58,
        "fp_score": 0.9990,
        "empty_accuracy": 0.85
    },
    "Z": {
        "threshold": 0.9,
        "min_area": 300,
        "fg_dice": 0.45,
        "fp_score": 0.9998,
        "empty_accuracy": 0.97
    }
}

def calculate_official_score(config):
    return (
        0.70 * config["fg_dice"]
        + 0.20 * config["fp_score"]
        + 0.10 * config["empty_accuracy"]
    )

# Calculate scores
scores = {}

for name, config in configs.items():
    scores[name] = calculate_official_score(config)

    print(
        f"Config {name}: "
        f"Official Score = {scores[name]:.4f}"
    )

# Select the best configuration
best_config = max(scores, key=scores.get)

print("\nBest configuration:", best_config)

# Compare against the configuration with highest FgDice
highest_fg_config = max(
    configs,
    key=lambda name: configs[name]["fg_dice"]
)

print("Highest FgDice configuration:", highest_fg_config)

best = configs[best_config]
highest_fg = configs[highest_fg_config]

print("\nWhy the selected configuration wins:")

print(
    "FalsePositiveScore improvement:",
    best["fp_score"] - highest_fg["fp_score"]
)

print(
    "EmptyAccuracy improvement:",
    best["empty_accuracy"] - highest_fg["empty_accuracy"]
)

# Final answer
print("\nFinal Answer:")
print(
    f"{scores['X']:.4f}, "
    f"{scores['Y']:.4f}, "
    f"{scores['Z']:.4f}, "
    f"{best_config}, "
    "FalsePositiveScore, EmptyAccuracy"
)

Config X: Official Score = 0.6730
Config Y: Official Score = 0.6908
Config Z: Official Score = 0.6120

Best configuration: Y
Highest FgDice configuration: X

Why the selected configuration wins:
FalsePositiveScore improvement: 0.0040000000000000036
EmptyAccuracy improvement: 0.44999999999999996

Final Answer:
0.6730, 0.6908, 0.6120, Y, FalsePositiveScore, EmptyAccuracy


In [3]:
import numpy as np
import cv2

# Given 6 x 8 binary prediction mask
mask = np.array([
    [1, 1, 0, 0, 0, 0, 0, 1],
    [1, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 1, 0],
    [1, 0, 0, 0, 0, 1, 1, 1]
], dtype=np.uint8)


def analyze_components(mask, connectivity):
    num_labels, labels, stats, centroids = (
        cv2.connectedComponentsWithStats(
            mask,
            connectivity=connectivity
        )
    )

    # Exclude background (label 0)
    component_areas = stats[1:, cv2.CC_STAT_AREA]

    return num_labels - 1, component_areas


# 1. Count components using 8-connectivity and 4-connectivity
n_8conn, areas_8 = analyze_components(mask, 8)
n_4conn, areas_4 = analyze_components(mask, 4)

# 2. Remove components with area < 5 using 8-connectivity
pixels_N5 = sum(area for area in areas_8 if area >= 5)

# 3. Remove components with area < 6 using 8-connectivity
pixels_N6 = sum(area for area in areas_8 if area >= 6)

# Display results
print("Original mask:")
print(mask)

print("\n8-connectivity component areas:", areas_8)
print("4-connectivity component areas:", areas_4)

print("\nNumber of components (8-connectivity):", n_8conn)
print("Number of components (4-connectivity):", n_4conn)

print("Foreground pixels remaining (area >= 5):", pixels_N5)
print("Foreground pixels remaining (area >= 6):", pixels_N6)

print("\nFinal Answer:")
print(f"{n_8conn}, {n_4conn}, {pixels_N5}, {pixels_N6}")

Original mask:
[[1 1 0 0 0 0 0 1]
 [1 1 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0 0]
 [0 0 0 0 0 1 1 0]
 [1 0 0 0 0 1 1 1]]

8-connectivity component areas: [6 1 1 5]
4-connectivity component areas: [4 1 1 1 5 1]

Number of components (8-connectivity): 4
Number of components (4-connectivity): 6
Foreground pixels remaining (area >= 5): 11
Foreground pixels remaining (area >= 6): 6

Final Answer:
4, 6, 11, 6


In [4]:
# Q4: Does Post-Processing Actually Improve the Official Score?

# Dataset information
total_pairs = 100
positive_pairs = 20
empty_pairs = 80

# Positive pair Dice scores before filtering
positive_dice = [0.70] * 12 + [0.45] * 8

# Empty pair distribution
already_empty = 50
small_speckles = 20
large_blobs = 10

# Apply minimum-area filtering (N = 50)
# All speckles are removed.
# Large blobs remain.
# One positive pair loses its only predicted component.

positive_dice_after = positive_dice.copy()
positive_dice_after[12] = 0.0

# 1. Foreground Dice after filtering
fg_dice_after = sum(positive_dice_after) / positive_pairs

# 2. FalsePositiveScore
canvas_pixels = 128 * 800
fp_pixels_after = large_blobs * 2000

fp_after = 1 - (fp_pixels_after / (empty_pairs * canvas_pixels))

# 3. EmptyAccuracy
correct_empty_after = already_empty + small_speckles
empty_acc_after = correct_empty_after / empty_pairs

# 4. Official score before filtering
# Before filtering:
# Positive Dice = mean of original positive pair Dice scores
fg_dice_before = sum(positive_dice) / positive_pairs

# FP pixels before filtering:
fp_pixels_before = (
    small_speckles * 20 +
    large_blobs * 2000
)

fp_before = 1 - (fp_pixels_before / (empty_pairs * canvas_pixels))

# Only 50 empty pairs were already empty before filtering
empty_acc_before = already_empty / empty_pairs

score_before = (
    0.70 * fg_dice_before
    + 0.20 * fp_before
    + 0.10 * empty_acc_before
)

# Official score after filtering
score_after = (
    0.70 * fg_dice_after
    + 0.20 * fp_after
    + 0.10 * empty_acc_after
)

# Display results
print(f"ForegroundDice after: {fg_dice_after:.4f}")
print(f"FalsePositiveScore after: {fp_after:.4f}")
print(f"EmptyAccuracy after: {empty_acc_after:.4f}")

print(f"\nOfficial Score before filtering: {score_before:.4f}")
print(f"Official Score after filtering:  {score_after:.4f}")

print("\nFinal Answer:")
print(
    f"{fg_dice_after:.4f}, "
    f"{fp_after:.4f}, "
    f"{empty_acc_after:.4f}, "
    f"{score_before:.4f}, "
    f"{score_after:.4f}"
)

ForegroundDice after: 0.5775
FalsePositiveScore after: 0.9976
EmptyAccuracy after: 0.8750

Official Score before filtering: 0.6820
Official Score after filtering:  0.6913

Final Answer:
0.5775, 0.9976, 0.8750, 0.6820, 0.6913


In [6]:
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
from collections import Counter

# ============================================================
# 1. LOAD DATASET
# ============================================================

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "public"
TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_IMAGES = DATA_DIR / "train"
TEST_IMAGES = DATA_DIR / "test"

assert TRAIN_CSV.exists(), f"Missing CSV: {TRAIN_CSV}"
assert TRAIN_IMAGES.is_dir(), f"Missing training images: {TRAIN_IMAGES}"

df = pd.read_csv(TRAIN_CSV)

print("Dataset shape:", df.shape)
print("Unique image IDs:", df["image_id"].nunique())
print("Defect classes:", sorted(df["defect_type"].dropna().unique()))


# ============================================================
# 2. CONFIGURATION
# ============================================================

HEIGHT = 128
WIDTH = 800
TOTAL_PIXELS = HEIGHT * WIDTH

MIN_AREA = 100

CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]


# ============================================================
# 3. RLE DECODING
# ============================================================

def decode_rle(rle, height=HEIGHT, width=WIDTH):

    mask = np.zeros(height * width, dtype=np.uint8)

    if pd.isna(rle) or str(rle).strip() == "":
        return mask.reshape((height, width), order="F")

    values = np.asarray(str(rle).split(), dtype=np.int64)

    starts = values[0::2] - 1
    lengths = values[1::2]

    for start, length in zip(starts, lengths):

        end = min(start + length, height * width)

        if start < height * width and end > start:
            mask[start:end] = 1

    return mask.reshape((height, width), order="F")


# ============================================================
# 4. CONNECTED COMPONENT ANALYSIS
# ============================================================

wiped_by_class = Counter()

total_positive_pairs = 0
total_wiped_pairs = 0

for _, row in df.iterrows():

    rle = row["mask_rle"]
    defect_class = row["defect_type"]

    # Consider only positive image-class pairs
    if pd.isna(rle) or str(rle).strip() == "":
        continue

    total_positive_pairs += 1

    # Decode ground-truth mask
    mask = decode_rle(rle)

    # Find 8-connected components
    num_labels, labels, stats, centroids = (
        cv2.connectedComponentsWithStats(
            mask,
            connectivity=8
        )
    )

    # Exclude background component
    component_areas = stats[1:, cv2.CC_STAT_AREA]

    # Pair is wiped out if every component has area < 100
    if len(component_areas) == 0 or np.all(component_areas < MIN_AREA):

        total_wiped_pairs += 1
        wiped_by_class[defect_class] += 1


# ============================================================
# 5. FIND MOST AFFECTED DEFECT CLASS
# ============================================================

most_affected_class = max(
    CLASSES,
    key=lambda cls: wiped_by_class[cls]
)


# ============================================================
# 6. DISPLAY RESULTS
# ============================================================

print("\n========== Q5 RESULTS ==========")

print("Minimum component area:", MIN_AREA)
print("Connectivity: 8")

print("\nTotal positive image-class pairs:", total_positive_pairs)
print("Wiped-out positive pairs:", total_wiped_pairs)

print("\nWiped-out pairs by defect class:")

for cls in CLASSES:
    print(f"{cls}: {wiped_by_class[cls]}")

print("\nMost affected defect class:", most_affected_class)

print("\n========== FINAL ANSWER ==========")

print(f"{total_wiped_pairs}, {most_affected_class}")

Dataset shape: (45572, 5)
Unique image IDs: 11393
Defect classes: ['defect_a', 'defect_b', 'defect_c', 'defect_d']

========== Q5 RESULTS ==========
Minimum component area: 100
Connectivity: 8

Total positive image-class pairs: 7201
Wiped-out positive pairs: 48

Wiped-out pairs by defect class:
defect_a: 22
defect_b: 1
defect_c: 24
defect_d: 1

Most affected defect class: defect_c

========== FINAL ANSWER ==========
48, defect_c


In [7]:
import torch

# ============================================================
# Q6: TEST-TIME AUGMENTATION
# ============================================================

# Input tensor layout: (B, C, H, W)
# Shape = (1, 1, 128, 800)
#
# We use a 2x2 patch to demonstrate the calculations.

original = torch.tensor([
    [[
        [0.9, 0.6],
        [0.4, 0.2]
    ]]
], dtype=torch.float32)

# Horizontal-flip pass (raw output in flipped frame)
horizontal_raw = torch.tensor([
    [[
        [0.5, 0.8],
        [0.3, 0.4]
    ]]
], dtype=torch.float32)

# Vertical-flip pass (raw output in flipped frame)
vertical_raw = torch.tensor([
    [[
        [0.1, 0.5],
        [0.7, 0.9]
    ]]
], dtype=torch.float32)


# ============================================================
# 1. FLIP DIMENSIONS
# ============================================================

# Tensor layout: (B, C, H, W)
horizontal_flip_dim = 3  # Flip width
vertical_flip_dim = 2    # Flip height

# ============================================================
# 2. INVERSE TRANSFORMATIONS
# ============================================================

# Undo horizontal flip
horizontal_corrected = torch.flip(
    horizontal_raw,
    dims=[horizontal_flip_dim]
)

# Undo vertical flip
vertical_corrected = torch.flip(
    vertical_raw,
    dims=[vertical_flip_dim]
)

# ============================================================
# 3. CORRECT TTA: INVERSE FLIP THEN AVERAGE
# ============================================================

correct_average = (
    original +
    horizontal_corrected +
    vertical_corrected
) / 3.0

# Top-right pixel
top_right_value = correct_average[0, 0, 0, 1].item()

# Threshold using strict > 0.5
correct_mask = (correct_average > 0.5).int()

n_correct = correct_mask.sum().item()


# ============================================================
# 4. BUGGY TTA: AVERAGE WITHOUT INVERSE FLIPS
# ============================================================

buggy_average = (
    original +
    horizontal_raw +
    vertical_raw
) / 3.0

buggy_mask = (buggy_average > 0.5).int()

n_buggy = buggy_mask.sum().item()


# ============================================================
# 5. DISPLAY RESULTS
# ============================================================

print("Original:")
print(original[0, 0])

print("\nHorizontal corrected:")
print(horizontal_corrected[0, 0])

print("\nVertical corrected:")
print(vertical_corrected[0, 0])

print("\nCorrect averaged probabilities:")
print(correct_average[0, 0])

print(f"\nTop-right averaged probability: {top_right_value:.4f}")
print("Positive pixels with correct TTA:", n_correct)

print("\nBuggy averaged probabilities (without inverse flips):")
print(buggy_average[0, 0])

print("Positive pixels with buggy TTA:", n_buggy)

print("\n========== FINAL ANSWER ==========")
print(
    f"{horizontal_flip_dim}, "
    f"{vertical_flip_dim}, "
    f"{top_right_value:.4f}, "
    f"{n_correct}, "
    f"{n_buggy}"
)

Original:
tensor([[0.9000, 0.6000],
        [0.4000, 0.2000]])

Horizontal corrected:
tensor([[0.8000, 0.5000],
        [0.4000, 0.3000]])

Vertical corrected:
tensor([[0.7000, 0.9000],
        [0.1000, 0.5000]])

Correct averaged probabilities:
tensor([[0.8000, 0.6667],
        [0.3000, 0.3333]])

Top-right averaged probability: 0.6667
Positive pixels with correct TTA: 2

Buggy averaged probabilities (without inverse flips):
tensor([[0.5000, 0.6333],
        [0.4667, 0.5000]])
Positive pixels with buggy TTA: 1

========== FINAL ANSWER ==========
3, 2, 0.6667, 2, 1


In [8]:
import numpy as np
from scipy.special import expit

# ============================================================
# Q7: ENSEMBLING
# ============================================================

# Ground truth: positive at p1, p3, p5
y_true = np.array([1, 0, 1, 0, 1])

# Predictions from three models
model_A = np.array([0.90, 0.55, 0.80, 0.10, 0.60])
model_B = np.array([0.40, 0.55, 0.70, 0.20, 0.45])
model_C = np.array([0.45, 0.20, 0.90, 0.30, 0.55])

T = 0.5

# ============================================================
# 1. SOFT ENSEMBLE
# ============================================================

# Average probabilities first, then threshold
soft_probabilities = np.mean(
    [model_A, model_B, model_C],
    axis=0
)

soft_predictions = (soft_probabilities > T).astype(int)


# ============================================================
# 2. HARD-VOTE ENSEMBLE
# ============================================================

# Threshold each model separately
individual_predictions = np.array([
    model_A > T,
    model_B > T,
    model_C > T
]).astype(int)

# Majority voting
hard_predictions = (
    individual_predictions.sum(axis=0) >= 2
).astype(int)


# ============================================================
# 3. DICE CALCULATION
# ============================================================

def calculate_dice(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    denominator = 2 * tp + fp + fn

    return 2 * tp / denominator if denominator > 0 else 1.0


dice_soft = calculate_dice(y_true, soft_predictions)
dice_vote = calculate_dice(y_true, hard_predictions)


# ============================================================
# 4. WEIGHTED ENSEMBLE
# ============================================================

weights = np.array([0.5, 0.3, 0.2])

weighted_p1 = (
    weights[0] * model_A[0]
    + weights[1] * model_B[0]
    + weights[2] * model_C[0]
)


# ============================================================
# 5. PROBABILITY-SPACE VS LOGIT-SPACE AVERAGING
# ============================================================

# Two snapshots produce logits +1 and -3
logits = np.array([1.0, -3.0])

# Method 1: Convert logits to probabilities, then average
snapshot_probabilities = expit(logits)

prob_avg = np.mean(snapshot_probabilities)

# Method 2: Average logits first, then apply sigmoid
logit_avg = expit(np.mean(logits))

# Threshold T = 0.3
method = (
    "prob_avg"
    if prob_avg > 0.3
    else "logit_avg"
)


# ============================================================
# 6. DISPLAY RESULTS
# ============================================================

print("Soft ensemble probabilities:")
print(np.round(soft_probabilities, 4))

print("\nSoft ensemble predictions:")
print(soft_predictions)

print("\nHard-vote predictions:")
print(hard_predictions)

print(f"\nDice of soft ensemble: {dice_soft:.4f}")
print(f"Dice of hard-vote ensemble: {dice_vote:.4f}")

print(f"\nWeighted ensemble probability at p1: {weighted_p1:.4f}")

print("\nSnapshot probabilities:")
print(np.round(snapshot_probabilities, 4))

print(f"Probability-space average: {prob_avg:.4f}")
print(f"Logit-space average: {logit_avg:.4f}")
print("Positive prediction at T=0.3:", method)

print("\n========== FINAL ANSWER ==========")

print(
    f"{dice_soft:.4f}, "
    f"{dice_vote:.4f}, "
    f"{weighted_p1:.4f}, "
    f"{prob_avg:.4f}, "
    f"{logit_avg:.4f}, "
    f"{method}"
)

Soft ensemble probabilities:
[0.5833 0.4333 0.8    0.2    0.5333]

Soft ensemble predictions:
[1 0 1 0 1]

Hard-vote predictions:
[0 1 1 0 1]

Dice of soft ensemble: 1.0000
Dice of hard-vote ensemble: 0.6667

Weighted ensemble probability at p1: 0.6600

Snapshot probabilities:
[0.7311 0.0474]
Probability-space average: 0.3892
Logit-space average: 0.2689
Positive prediction at T=0.3: prob_avg

========== FINAL ANSWER ==========
1.0000, 0.6667, 0.6600, 0.3892, 0.2689, prob_avg


In [9]:
import math

# ============================================================
# Q8: INFERENCE MEMORY AND TIME BUDGET
# ============================================================

# Dataset information
N_IMAGES = 5000
HEIGHT = 128
WIDTH = 800
N_CLASSES = 4

# Model inference time
TIME_PER_FORWARD_PASS_MS = 12

# Available resources
RAM_BUDGET_GB = 16
TOTAL_TIME_LIMIT_MIN = 30
DATA_LOADING_AND_IO_MIN = 8

# ============================================================
# 1. PROBABILITY MAP MEMORY
# ============================================================

TOTAL_VALUES = N_IMAGES * N_CLASSES * HEIGHT * WIDTH

BYTES_PER_GB = 10**9

GB_F32 = TOTAL_VALUES * 4 / BYTES_PER_GB
GB_F16 = TOTAL_VALUES * 2 / BYTES_PER_GB
GB_U8 = TOTAL_VALUES * 1 / BYTES_PER_GB

# ============================================================
# 2. CONFIGURATION MEMORY REQUIREMENTS
# ============================================================

# Config A: 3 models, 3-view TTA
MODELS_A = 3
VIEWS_A = 3

# Config B: 2 models, 2-view TTA
MODELS_B = 2
VIEWS_B = 2

# Naively store every prediction as float32
GB_A_NAIVE = MODELS_A * VIEWS_A * GB_F32
GB_B_NAIVE = MODELS_B * VIEWS_B * GB_F32

# Determine whether each configuration fits in 16 GB
fits_A = GB_A_NAIVE <= RAM_BUDGET_GB
fits_B = GB_B_NAIVE <= RAM_BUDGET_GB

if fits_A and fits_B:
    memory_fit = "both"
elif fits_A:
    memory_fit = "A"
elif fits_B:
    memory_fit = "B"
else:
    memory_fit = "neither"

# Running-sum accumulation requires one float32 map
ACCUMULATOR_GB = GB_F32

# ============================================================
# 3. FORWARD PASSES AND INFERENCE TIME
# ============================================================

PASSES_A = N_IMAGES * MODELS_A * VIEWS_A
PASSES_B = N_IMAGES * MODELS_B * VIEWS_B

MIN_A = PASSES_A * TIME_PER_FORWARD_PASS_MS / 60000
MIN_B = PASSES_B * TIME_PER_FORWARD_PASS_MS / 60000

# ============================================================
# 4. MAXIMUM NUMBER OF MODELS WITHIN TIME LIMIT
# ============================================================

INFERENCE_BUDGET_MIN = (
    TOTAL_TIME_LIMIT_MIN - DATA_LOADING_AND_IO_MIN
)

# Time required per model
MIN_PER_MODEL_3VIEW = (
    N_IMAGES * 3 * TIME_PER_FORWARD_PASS_MS / 60000
)

MIN_PER_MODEL_2VIEW = (
    N_IMAGES * 2 * TIME_PER_FORWARD_PASS_MS / 60000
)

MAX_MODELS_3VIEW = math.floor(
    INFERENCE_BUDGET_MIN / MIN_PER_MODEL_3VIEW
)

MAX_MODELS_2VIEW = math.floor(
    INFERENCE_BUDGET_MIN / MIN_PER_MODEL_2VIEW
)

# ============================================================
# 5. DISPLAY RESULTS
# ============================================================

print("========== PROBABILITY MAP MEMORY ==========")
print(f"Float32: {GB_F32:.3f} GB")
print(f"Float16: {GB_F16:.3f} GB")
print(f"Uint8:   {GB_U8:.3f} GB")

print("\n========== NAIVE MEMORY REQUIREMENTS ==========")
print(f"Config A: {GB_A_NAIVE:.3f} GB")
print(f"Config B: {GB_B_NAIVE:.3f} GB")
print(f"Fits in 16 GB RAM: {memory_fit}")

print("\n========== RUNNING-SUM ACCUMULATION ==========")
print(f"Accumulator memory: {ACCUMULATOR_GB:.3f} GB")

print("\n========== FORWARD PASSES AND TIME ==========")
print(f"Config A passes: {PASSES_A}")
print(f"Config B passes: {PASSES_B}")
print(f"Config A inference time: {MIN_A:.2f} minutes")
print(f"Config B inference time: {MIN_B:.2f} minutes")

print("\n========== MAXIMUM ENSEMBLE SIZE ==========")
print(f"Available inference time: {INFERENCE_BUDGET_MIN} minutes")
print(f"Maximum models with 3-view TTA: {MAX_MODELS_3VIEW}")
print(f"Maximum models with 2-view TTA: {MAX_MODELS_2VIEW}")

# ============================================================
# 6. FINAL ANSWER
# ============================================================

print("\n========== FINAL ANSWER ==========")

print(
    f"{GB_F32:.3f}, "
    f"{GB_F16:.3f}, "
    f"{GB_U8:.3f}, "
    f"{GB_A_NAIVE:.3f}, "
    f"{GB_B_NAIVE:.3f}, "
    f"{memory_fit}, "
    f"{ACCUMULATOR_GB:.3f}, "
    f"{PASSES_A}, "
    f"{PASSES_B}, "
    f"{MIN_A:.1f}, "
    f"{MIN_B:.1f}, "
    f"{MAX_MODELS_3VIEW}, "
    f"{MAX_MODELS_2VIEW}"
)

========== PROBABILITY MAP MEMORY ==========
Float32: 8.192 GB
Float16: 4.096 GB
Uint8:   2.048 GB

========== NAIVE MEMORY REQUIREMENTS ==========
Config A: 73.728 GB
Config B: 32.768 GB
Fits in 16 GB RAM: neither

========== RUNNING-SUM ACCUMULATION ==========
Accumulator memory: 8.192 GB

========== FORWARD PASSES AND TIME ==========
Config A passes: 45000
Config B passes: 20000
Config A inference time: 9.00 minutes
Config B inference time: 4.00 minutes

========== MAXIMUM ENSEMBLE SIZE ==========
Available inference time: 22 minutes
Maximum models with 3-view TTA: 7
Maximum models with 2-view TTA: 11

========== FINAL ANSWER ==========
8.192, 4.096, 2.048, 73.728, 32.768, neither, 8.192, 45000, 20000, 9.0, 4.0, 7, 11


### 1. What should be the correct order of the inference pipeline?

I would start by loading the test images and applying the same preprocessing steps that I used during training. This includes resizing the images to the required dimensions, converting them into tensors, and normalizing them.

Next, I would load the trained model checkpoints and put the models in evaluation mode. For better predictions, I would use Test-Time Augmentation (TTA), such as horizontal and vertical flips, along with the original image.

One important thing to remember is that after applying flips, the predicted masks must also be flipped back to their original positions. Otherwise, the predictions will not be properly aligned.

Once all the predictions are aligned, I would combine the probability maps from different models and TTA views by averaging them. After that, I would apply the selected probability threshold for each defect class to obtain binary masks.

Then I would use connected-component filtering to remove very small regions that are likely to be false positives. Finally, I would convert the processed masks into RLE format and generate the submission CSV with the required columns: `image_id`, `defect_type`, and `mask_rle`.

The order is important because each step depends on the output of the previous one. For example, averaging should happen before thresholding so that we do not lose useful probability information. Similarly, inverse flipping must happen before averaging so that all predictions refer to the same pixel locations.

### 2. How can we avoid overfitting while selecting thresholds and minimum-area values?

I would use a separate validation dataset to select the thresholds and minimum-area values instead of tuning them directly on the training data. This gives a better idea of how the selected values may perform on unseen images.

I would also avoid choosing values just because they give the highest score on one validation run. Instead, I would compare the results across different validation splits and select values that give reasonably consistent performance.

Another important point is that this dataset contains many empty image-class pairs. So, simply predicting empty masks may sometimes give a misleadingly good score. If the threshold or minimum-area value is too aggressive, small but genuine defects may also get removed.

Therefore, I would consider ForegroundDice, FalsePositiveScore, and EmptyAccuracy together while selecting the final parameters, rather than focusing only on one metric.

### 3. How should empty masks be handled, and what checks should be performed before submission?

If the model does not detect any defect in an image-class pair, the final binary mask will be completely empty. In that case, I would store an empty string in the `mask_rle` column instead of trying to encode a mask that has no foreground pixels.

Before submitting, I would perform two important checks:

1. **Check the submission format:** I would verify that every expected image-class pair is present exactly once, there are no duplicate or missing rows, and the CSV contains the required columns. I would also check that the RLE values are valid.

2. **Check predictions and resource usage:** I would verify that all predicted masks have the expected dimensions and contain only 0s and 1s. I would also test the inference notebook on a small batch first and monitor its memory usage and execution time to make sure it can complete within Kaggle's limits.

These checks help avoid common mistakes such as incorrect RLE encoding, missing predictions, invalid masks, or the notebook running out of memory during inference.